# Learn Exploratory Data Analysis (EDA) with the Instacart Dataset

**What you will learn in this notebook:**

| # | Topic | Key Skills |
|---|-------|-----------|
| 1 | Loading & inspecting data | `pd.read_csv`, `.shape`, `.head()`, `.dtypes` |
| 2 | Summary statistics | `.describe()`, `.value_counts()` |
| 3 | Missing values | `.isna()`, `.isnull()`, heatmaps |
| 4 | Table relationships | understanding relational data, foreign keys |
| 5 | Univariate analysis | histograms, bar charts |
| 6 | Merging tables | `pd.merge()`, enriching data |
| 7 | Categorical analysis | value counts, proportions, bar charts |
| 8 | Bivariate analysis | grouped aggregations, scatter plots |
| 9 | Time-based patterns | day-of-week & hour-of-day analysis |
| 10 | Correlation & cross-tabulation | `.corr()`, `pd.crosstab()`, heatmaps |
| 11 | Practice exercises | test your skills! |

> **Dataset:** [The Instacart Online Grocery Shopping Dataset 2017](https://www.instacart.com/datasets/grocery-shopping-2017) — over 3 million grocery orders from 200,000+ users.

All visualizations use **Plotly Express** with the `simple_white` template.

## Setup — Import Libraries

Before we begin, we need to import the libraries we will use throughout this notebook.

- **pandas** — the go-to library for tabular data manipulation in Python.
- **plotly.express** — a high-level library for creating beautiful, interactive charts with very little code.

In [ ]:
import os
import zipfile

import pandas as pd
import plotly.express as px

# Use a clean, minimal chart style for all plots
px.defaults.template = "simple_white"

---
## Step 1 — Load and Inspect the Data

The first step in any EDA is to **load the data and get a feel for its size and structure**.

The Instacart dataset is stored as several CSV files. Let's load them all and see what we're working with.

> 💡 **Tip:** Always start by checking the **shape** (rows × columns) and the **first few rows** of each table.

In [ ]:
# The orders file is compressed — unzip it if necessary
if not os.path.exists("orders.csv"):
    with zipfile.ZipFile("orders.csv.zip", "r") as z:
        z.extractall(".")

# Load all five tables
orders         = pd.read_csv("orders.csv")
order_products = pd.read_csv("order_products_train.csv")
products       = pd.read_csv("products.csv")
aisles         = pd.read_csv("aisles.csv")
departments    = pd.read_csv("departments.csv")

# Print the shape of each table
for name, df in [("orders", orders), ("order_products", order_products),
                 ("products", products), ("aisles", aisles),
                 ("departments", departments)]:
    print(f"{name:20s}  →  {df.shape[0]:>10,} rows × {df.shape[1]} columns")

### 1a — Peek at each table with `.head()`

`.head()` shows the first 5 rows. It is the quickest way to see what the columns contain.

In [ ]:
orders.head()

In [ ]:
order_products.head()

In [ ]:
products.head()

### 1b — Data types with `.dtypes`

Understanding column types is critical. Numeric columns can be aggregated; categorical/object columns need special treatment.

> 💡 **Tip:** If a column you expect to be numeric shows as `object`, it may contain unexpected text or missing-value markers.

In [ ]:
orders.dtypes

### 1c — Quick summary with `.info()`

`.info()` gives you column names, non-null counts, and data types all in one shot — very handy!

In [ ]:
orders.info()

---
## Step 2 — Summary Statistics with `.describe()`

`.describe()` computes count, mean, standard deviation, min, quartiles, and max for every numeric column.

This is your first look at the **distribution** of the data — are there outliers? What is the typical range?

In [ ]:
orders.describe()

**What to look for:**

- **count** — does it match the number of rows? If not, there are missing values.
- **min / max** — are the ranges reasonable?
- **mean vs. median (50%)** — a large gap indicates a skewed distribution.
- **std** — high standard deviation means high variability.

In [ ]:
order_products.describe()

---
## Step 3 — Checking for Missing Values

Missing values can distort your analysis. Always check for them early.

We'll count missing values per column and visualize them.

In [ ]:
# Count missing values in each table
for name, df in [("orders", orders), ("order_products", order_products),
                 ("products", products), ("aisles", aisles),
                 ("departments", departments)]:
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) == 0:
        print(f"✅ {name}: no missing values")
    else:
        print(f"⚠️  {name}:")
        for col, count in missing.items():
            pct = 100 * count / len(df)
            print(f"     {col}: {count:,} missing ({pct:.1f}%)")

> 💡 **Why is `days_since_prior_order` missing?** Because a customer's *very first order* has no prior order — so the field is naturally `NaN`. This is **expected**, not a data-quality issue.

In [ ]:
# Visualize the proportion of missing values
missing_pct = (
    orders.isnull().mean() * 100
).reset_index()
missing_pct.columns = ["Column", "% Missing"]

fig = px.bar(
    missing_pct,
    x="Column",
    y="% Missing",
    title="Percentage of Missing Values in the Orders Table",
    text_auto=".1f",
)
fig.update_layout(yaxis_range=[0, max(missing_pct["% Missing"].max() * 1.2, 1)])
fig.show()

---
## Step 4 — Understanding Table Relationships

Real-world data rarely lives in a single table. The Instacart dataset has **five related tables**:

```
departments  ──┐
               ├──  products  ──┐
aisles  ───────┘                ├──  order_products  ──── orders
                                │
                          (product_id)              (order_id)
```

- **orders** — one row per order; contains user and timing info.
- **order_products** — one row per *item in an order*; links `order_id` → `product_id`.
- **products** — one row per product; links to `aisle_id` and `department_id`.
- **aisles** — lookup table for aisle names.
- **departments** — lookup table for department names.

> 💡 **Key concept:** A *foreign key* (e.g., `product_id` in `order_products`) references the *primary key* of another table (`product_id` in `products`). This is how tables are joined.

In [ ]:
# Verify the linking columns exist and check their cardinality
print("Unique order_ids in order_products:", order_products["order_id"].nunique())
print("Unique order_ids in orders:        ", orders["order_id"].nunique())
print()
print("Unique product_ids in order_products:", order_products["product_id"].nunique())
print("Unique product_ids in products:      ", products["product_id"].nunique())
print()
print("Unique aisle_ids in products:", products["aisle_id"].nunique())
print("Unique aisles:              ", len(aisles))
print()
print("Unique department_ids in products:", products["department_id"].nunique())
print("Unique departments:              ", len(departments))

---
## Step 5 — Univariate Analysis (One Variable at a Time)

Univariate analysis examines the distribution of a **single column**. The right chart depends on the data type:

| Data Type | Good Charts |
|-----------|------------|
| Continuous numeric | Histogram, box plot |
| Discrete numeric / ordinal | Bar chart |
| Categorical | Bar chart (value counts) |

Let's start with some important numeric distributions.

### 5a — Distribution of `order_hour_of_day`

In [ ]:
fig = px.histogram(
    orders,
    x="order_hour_of_day",
    nbins=24,
    title="When Do Customers Place Orders? (Hour of Day)",
    labels={"order_hour_of_day": "Hour of Day (0–23)", "count": "Number of Orders"},
)
fig.show()

> **Observation:** Most orders are placed between 9 AM and 5 PM, peaking around 10 AM. Very few orders happen in the early morning hours.

### 5b — Distribution of `days_since_prior_order`

In [ ]:
fig = px.histogram(
    orders["days_since_prior_order"].dropna(),
    nbins=31,
    title="How Many Days Between Consecutive Orders?",
    labels={"value": "Days Since Prior Order", "count": "Number of Orders"},
)
fig.show()

> **Observation:** There's a strong spike at **30 days** (the maximum in this dataset) and another spike at **7 days**, suggesting weekly shoppers.

### 5c — Distribution of `order_number` (how many orders a customer has placed)

In [ ]:
fig = px.histogram(
    orders,
    x="order_number",
    nbins=100,
    title="Distribution of Order Sequence Number",
    labels={"order_number": "Order Number (nth order for user)", "count": "Number of Orders"},
)
fig.show()

### 5d — Distribution of `add_to_cart_order`

In [ ]:
fig = px.histogram(
    order_products,
    x="add_to_cart_order",
    nbins=50,
    title="Distribution of Add-to-Cart Position",
    labels={"add_to_cart_order": "Position Added to Cart", "count": "Count"},
)
fig.show()

> **Observation:** Most items are added in the first 10–15 positions. Carts rarely exceed 30–40 items.

---
## Step 6 — Merging Tables to Enrich the Data

Individual tables have IDs, but we want **names**. Merging (joining) lets us combine information across tables.

```python
pd.merge(left_df, right_df, on="shared_column")
```

We'll build an **enriched order-products table** that includes product names, aisle names, and department names.

In [ ]:
# Step-by-step merge
op = (
    order_products
    .merge(products, on="product_id")       # add product_name, aisle_id, department_id
    .merge(aisles, on="aisle_id")            # add aisle name
    .merge(departments, on="department_id")  # add department name
)

print(f"Enriched table shape: {op.shape}")
op.head()

> 💡 **Tip:** After merging, always check the shape and a few rows to confirm the join worked as expected. The row count should stay the same if you're doing a many-to-one join (which we are here).

---
## Step 7 — Categorical Analysis (Value Counts & Proportions)

For categorical columns, **value counts** reveal which categories dominate.

> 💡 `.value_counts()` returns categories sorted by frequency — most common first.

### 7a — Top 20 Products

In [ ]:
top20 = op["product_name"].value_counts().head(20).reset_index()
top20.columns = ["Product", "Times Ordered"]

fig = px.bar(
    top20,
    x="Times Ordered",
    y="Product",
    orientation="h",
    title="Top 20 Most Ordered Products",
    text_auto=True,
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

### 7b — Orders by Department

In [ ]:
dept_counts = op["department"].value_counts().reset_index()
dept_counts.columns = ["Department", "Items Ordered"]

fig = px.bar(
    dept_counts,
    x="Items Ordered",
    y="Department",
    orientation="h",
    title="Number of Items Ordered by Department",
    text_auto=True,
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

### 7c — Proportion of reordered vs. first-time items

In [ ]:
reorder_counts = order_products["reordered"].value_counts().reset_index()
reorder_counts.columns = ["Reordered", "Count"]
reorder_counts["Reordered"] = reorder_counts["Reordered"].map({1: "Reordered", 0: "First time"})

fig = px.pie(
    reorder_counts,
    values="Count",
    names="Reordered",
    title="Reordered vs. First-Time Items",
)
fig.show()

---
## Step 8 — Bivariate Analysis (Two Variables Together)

Bivariate analysis explores the **relationship between two variables**. Common techniques:

| Technique | When to Use |
|-----------|------------|
| Grouped bar chart | Compare a metric across categories |
| Scatter plot | Two continuous variables |
| Box plot | Distribution of a numeric variable across categories |

### 8a — Reorder ratio by department

> **Question:** Which departments have the highest share of *repeat purchases*?

In [ ]:
reorder_dept = (
    op.groupby("department")["reordered"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
reorder_dept.columns = ["Department", "Reorder Ratio"]

fig = px.bar(
    reorder_dept,
    x="Reorder Ratio",
    y="Department",
    orientation="h",
    title="Reorder Ratio by Department (higher = more repeat purchases)",
    text_auto=".2f",
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

> **Insight:** Personal care, dairy/eggs, and beverages have the highest reorder rates — these are routine essentials.

### 8b — Average cart position by department

In [ ]:
cart_dept = (
    op.groupby("department")["add_to_cart_order"]
    .mean()
    .sort_values()
    .reset_index()
)
cart_dept.columns = ["Department", "Avg Cart Position"]

fig = px.bar(
    cart_dept,
    x="Avg Cart Position",
    y="Department",
    orientation="h",
    title="Average Add-to-Cart Position by Department",
    text_auto=".1f",
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

> **Insight:** Items from *personal care* and *babies* tend to be added first (low position number), while *bulk* and *other* items are added later.

### 8c — Distribution of basket sizes

In [ ]:
basket_sizes = (
    order_products
    .groupby("order_id")
    .size()
    .reset_index(name="num_items")
)

fig = px.box(
    basket_sizes,
    y="num_items",
    title="Box Plot of Items per Order (Basket Size)",
    labels={"num_items": "Number of Items"},
)
fig.show()

> **Observation:** The median basket has about 8 items, but the distribution is right-skewed with some large orders exceeding 50 items.

---
## Step 9 — Time-Based Patterns

Time-based analysis is extremely common in EDA. We'll look at ordering patterns across days and hours.

### 9a — Orders by day of week

In [ ]:
dow_map = {0: "Saturday", 1: "Sunday", 2: "Monday", 3: "Tuesday",
           4: "Wednesday", 5: "Thursday", 6: "Friday"}
dow_counts = (
    orders["order_dow"]
    .map(dow_map)
    .value_counts()
    .reindex([dow_map[i] for i in range(7)])
    .reset_index()
)
dow_counts.columns = ["Day of Week", "Number of Orders"]

fig = px.bar(
    dow_counts,
    x="Day of Week",
    y="Number of Orders",
    title="Orders by Day of Week",
    text_auto=True,
)
fig.show()

> **Observation:** Saturday and Sunday see the most orders — grocery shopping is a weekend activity!

### 9b — Orders by hour of day, colored by weekend vs. weekday

In [ ]:
orders_time = orders.copy()
orders_time["day_type"] = orders_time["order_dow"].apply(
    lambda d: "Weekend" if d in [0, 1] else "Weekday"
)

hour_daytype = (
    orders_time
    .groupby(["order_hour_of_day", "day_type"])
    .size()
    .reset_index(name="count")
)

fig = px.line(
    hour_daytype,
    x="order_hour_of_day",
    y="count",
    color="day_type",
    title="Orders by Hour: Weekend vs Weekday",
    labels={"order_hour_of_day": "Hour of Day", "count": "Number of Orders",
            "day_type": "Day Type"},
    markers=True,
)
fig.show()

> **Observation:** Weekend orders start later and are more concentrated in the late morning. Weekday orders have a broader distribution.

### 9c — Heatmap: Day of week × Hour of day

In [ ]:
heatmap_data = (
    orders_time
    .assign(day_name=orders_time["order_dow"].map(dow_map))
    .groupby(["day_name", "order_hour_of_day"])
    .size()
    .reset_index(name="count")
    .pivot(index="day_name", columns="order_hour_of_day", values="count")
    .reindex([dow_map[i] for i in range(7)])
)

fig = px.imshow(
    heatmap_data,
    title="Order Heatmap: Day of Week × Hour of Day",
    labels=dict(x="Hour of Day", y="Day of Week", color="Orders"),
    aspect="auto",
    color_continuous_scale="Blues",
)
fig.show()

> **Insight:** The hottest spots are **Sunday 10 AM–3 PM** and **Saturday 10 AM–3 PM** — the prime grocery-shopping windows.

---
## Step 10 — Correlation and Cross-Tabulation

### 10a — Numeric correlation

Correlation measures the **linear relationship** between two numeric variables (−1 to +1).

In [ ]:
# Compute correlations for the orders table
orders_numeric = orders[["order_number", "order_dow", "order_hour_of_day",
                          "days_since_prior_order"]].dropna()
corr = orders_numeric.corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    title="Correlation Matrix — Orders Table",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    aspect="equal",
)
fig.show()

> **Reading the matrix:** Values close to 0 mean little linear relationship. The orders table columns are mostly independent of each other, which makes sense — when you order doesn't strongly predict how long you wait.

### 10b — Cross-tabulation

A **cross-tab** counts how often combinations of two categorical values occur. Let's see which aisles have the highest reorder rate.

In [ ]:
# Reorder rate by aisle (top 15 aisles by volume)
top15_aisles = op["aisle"].value_counts().head(15).index

aisle_reorder = (
    op[op["aisle"].isin(top15_aisles)]
    .groupby("aisle")["reordered"]
    .agg(["mean", "count"])
    .sort_values("mean", ascending=False)
    .reset_index()
)
aisle_reorder.columns = ["Aisle", "Reorder Rate", "Total Items"]

fig = px.scatter(
    aisle_reorder,
    x="Total Items",
    y="Reorder Rate",
    text="Aisle",
    title="Reorder Rate vs Volume for Top 15 Aisles",
    labels={"Total Items": "Number of Items Ordered", "Reorder Rate": "Reorder Rate"},
)
fig.update_traces(textposition="top center")
fig.show()

> **Insight:** Fresh fruits and fresh vegetables are both high-volume AND high-reorder — they are staple aisles. Packaged produce has high reorder rates with moderate volume.

---
## Step 11 — Practice Exercises 🏋️

Now it's your turn! Try these exercises using the skills you learned above.

**Exercise 1:** Find the top 10 most ordered *aisles* (not products). Create a horizontal bar chart.

**Exercise 2:** What percentage of all items in the dataset are reordered? Compute a single number.

**Exercise 3:** Create a histogram of *basket sizes* (number of items per order). What is the median basket size?

**Exercise 4:** Which *hour of day* has the highest reorder rate? Merge `order_products` with `orders` and group by `order_hour_of_day`.

**Exercise 5 (Challenge):** For the top 5 departments, create a grouped bar chart showing the reorder rate broken down by department *and* whether the item was added early (position 1–5) vs. late (position 6+) in the cart.

In [ ]:
# Exercise 1 — Your code here
# Hint: op["aisle"].value_counts().head(10)


In [ ]:
# Exercise 2 — Your code here
# Hint: order_products["reordered"].mean()


In [ ]:
# Exercise 3 — Your code here
# Hint: order_products.groupby("order_id").size()


In [ ]:
# Exercise 4 — Your code here
# Hint: merge order_products with orders on "order_id", then groupby


In [ ]:
# Exercise 5 (Challenge) — Your code here
# Hint: create an "early" column, then groupby(["department", "early"])


---
## Summary & Next Steps

Congratulations! You've completed an EDA walkthrough of the Instacart dataset. Here's a recap of the key techniques:

| Step | Technique | pandas / plotly Function |
|------|-----------|------------------------|
| Inspect | Shape, head, dtypes | `.shape`, `.head()`, `.dtypes`, `.info()` |
| Summarize | Descriptive stats | `.describe()` |
| Missing values | Null checks | `.isnull().sum()`, bar chart |
| Univariate | Single-variable distributions | `px.histogram()`, `px.bar()` |
| Merge | Combine tables | `pd.merge()` |
| Categorical | Value counts | `.value_counts()`, `px.bar()`, `px.pie()` |
| Bivariate | Two-variable relationships | `px.scatter()`, `px.box()`, grouped `.mean()` |
| Time patterns | Temporal analysis | `px.line()`, `px.imshow()` (heatmap) |
| Correlation | Linear relationships | `.corr()`, `px.imshow()` |

### What to explore next

- **Market basket analysis** — which products are frequently bought *together*? (See `customer_behavior_analysis.ipynb`)
- **Customer segmentation** — can you group users by their shopping behavior?
- **Predictive modeling** — can you predict whether a user will reorder a product?

> 📖 *"The Instacart Online Grocery Shopping Dataset 2017", Accessed from https://www.instacart.com/datasets/grocery-shopping-2017*